In [1]:
import pandas as pd

path = "../data/raw"
files = ["customers", "orders", "products", "order_items",
         "payments", "returns", "shipments"]

data = {name: pd.read_csv(f"{path}/{name}.csv") for name in files}

In [2]:
for name, df in data.items():
    print(f"\n{name}: rows={len(df)}, missing={df.isnull().sum().sum()}, duplicates={df.duplicated().sum()}")


customers: rows=25000, missing=2675, duplicates=0

orders: rows=100000, missing=116532, duplicates=0

products: rows=550, missing=4, duplicates=0

order_items: rows=254331, missing=0, duplicates=0

payments: rows=100000, missing=0, duplicates=0

returns: rows=12075, missing=0, duplicates=0

shipments: rows=89681, missing=4642, duplicates=0


In [3]:
for name, df in data.items():
    print(f"\n{name}:", df.dtypes.to_dict())


customers: {'customer_id': dtype('O'), 'customer_signup_date': dtype('O'), 'gender': dtype('O'), 'age': dtype('float64'), 'age_group': dtype('O'), 'state': dtype('O'), 'city': dtype('O'), 'pincode_prefix': dtype('int64'), 'customer_segment': dtype('O'), 'preferred_device': dtype('O'), 'preferred_payment_method': dtype('O'), 'acquisition_channel': dtype('O'), 'total_orders': dtype('int64'), 'total_spend': dtype('float64'), 'last_order_date': dtype('O'), 'average_order_value': dtype('float64'), 'customer_status': dtype('O'), 'loyalty_tier': dtype('O')}

orders: {'order_id': dtype('O'), 'customer_id': dtype('O'), 'order_date': dtype('O'), 'order_time': dtype('O'), 'order_status': dtype('O'), 'shipping_method': dtype('O'), 'delivery_city': dtype('O'), 'delivery_state': dtype('O'), 'coupon_code': dtype('O'), 'discount_percentage': dtype('float64'), 'marketing_channel': dtype('O'), 'subtotal': dtype('float64'), 'shipping_fee': dtype('float64'), 'tax_amount': dtype('float64'), 'final_amount'

In [4]:
for name in ["customers", "orders", "products"]:
    df = data[name]
    print(name, "rows:", len(df), "| duplicate IDs:", df.iloc[:, 0].duplicated().sum())

customers rows: 25000 | duplicate IDs: 0
orders rows: 100000 | duplicate IDs: 0
products rows: 550 | duplicate IDs: 0


In [5]:
for name in ["orders", "products", "order_items", "payments", "returns"]:
    df = data[name]
    cols = df.select_dtypes("number").columns
    print(f"\n{name}")
    print(df[cols].describe().round(2))


orders
       discount_percentage   subtotal  shipping_fee  tax_amount  final_amount
count            100000.00  100000.00     100000.00   100000.00     100000.00
mean                 11.98   24498.99          1.98     1224.95      25725.92
std                  10.69   42814.40         12.41     2140.72      44954.00
min                   0.00      33.50          0.00        1.68         71.00
25%                   0.00    3449.64          0.00      172.48       3622.12
50%                  10.00    9050.78          0.00      452.54       9503.31
75%                  20.00   24531.30          0.00     1226.56      25757.86
max                  30.00  578151.00        149.00    28907.55     607058.55

products
          price  cost_price  rating_average  rating_count  stock_quantity  \
count    550.00      550.00          546.00        550.00          550.00   
mean    4429.64     3531.20            4.01        126.94         1487.60   
std    10207.04     8786.80            0.43      

In [6]:
for name in ["orders", "products", "order_items"]:
    print(name, data[name].columns.tolist())

orders ['order_id', 'customer_id', 'order_date', 'order_time', 'order_status', 'shipping_method', 'delivery_city', 'delivery_state', 'coupon_code', 'discount_percentage', 'marketing_channel', 'subtotal', 'shipping_fee', 'tax_amount', 'final_amount', 'campaign_id']
products ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'price', 'cost_price', 'discount_range', 'rating_average', 'rating_count', 'stock_quantity', 'product_launch_date', 'product_type', 'return_rate_baseline']
order_items ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'discount_percentage', 'item_revenue', 'item_cost', 'profit']


In [7]:
for name, col in [
    ("customers", "customer_id"),
    ("orders", "order_id"),
    ("products", "product_id"),
    ("order_items", "order_item_id")
]:
    print(name, data[name][col].duplicated().sum())

customers 0
orders 0
products 0
order_items 0


In [8]:
for name, df in data.items():
    missing = df.isnull().sum()
    print(f"\n{name}:\n{missing[missing > 0]}")


customers:
age                  250
preferred_device     125
last_order_date     2300
dtype: int64

orders:
coupon_code    33711
campaign_id    82821
dtype: int64

products:
rating_average    4
dtype: int64

order_items:
Series([], dtype: int64)

payments:
Series([], dtype: int64)

returns:
Series([], dtype: int64)

shipments:
actual_delivery_date    2321
delivery_days           2321
dtype: int64


In [9]:
print("Order items without matching orders:",
      (~data["order_items"]["order_id"].isin(data["orders"]["order_id"])).sum())

print("Order items without matching products:",
      (~data["order_items"]["product_id"].isin(data["products"]["product_id"])).sum())

Order items without matching orders: 0
Order items without matching products: 0


In [10]:
data["order_items"][["quantity", "unit_price", "discount_percentage", "item_revenue"]].head()

,quantity,unit_price,discount_percentage,item_revenue
0,4,150.0,6.5,561.0
1,1,1150.0,5.0,1092.5
2,2,960.0,8.0,1766.4
3,4,920.0,5.0,3496.0
4,3,2640.0,5.0,7524.0


In [11]:
items = data["order_items"]

expected = items["quantity"] * items["unit_price"] * (
    1 - items["discount_percentage"] / 100
)

(items["item_revenue"] - expected).abs().describe().round(4)

count    254331.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64

In [12]:
for name, cols in {
    "customers": ["customer_id"],
    "orders": ["order_id", "customer_id", "order_date", "order_status"],
    "products": ["product_id", "price"],
    "order_items": ["order_item_id", "order_id", "product_id", "quantity", "unit_price"]
}.items():
    print(name, data[name][cols].isnull().sum().to_dict())

customers {'customer_id': 0}
orders {'order_id': 0, 'customer_id': 0, 'order_date': 0, 'order_status': 0}
products {'product_id': 0, 'price': 0}
order_items {'order_item_id': 0, 'order_id': 0, 'product_id': 0, 'quantity': 0, 'unit_price': 0}


In [13]:
items[["quantity", "unit_price", "discount_percentage", "item_revenue"]].agg(
    ["min", "max"]
)

,quantity,unit_price,discount_percentage,item_revenue
min,1,50.0,0.0,33.5
max,8,76020.0,33.0,577752.0


In [14]:
for name, col in [
    ("customers", "customer_id"),
    ("orders", "order_id"),
    ("products", "product_id"),
    ("order_items", "order_item_id")
]:
    print(name, data[name][col].duplicated().sum())

customers 0
orders 0
products 0
order_items 0


In [15]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

for name in ["customers", "orders", "products", "order_items"]:
    data[name].to_csv(f"../data/processed/{name}.csv", index=False)